# Walmart Weekly Sales — корректный временной backtest на Spark

Ноутбук целиком формирует `artifacts/regression/walmart/report.json` в схеме, которую использует ML Regression Console.

**Принцип оценки:** строки упорядочиваются по дате; модели обучаются только на раннем историческом отрезке, а метрики и прогнозы считаются на более позднем holdout. В UI горизонт 4 / 8 / 13 недель выбирает последние N недель этого holdout: на графике остаётся вся фактическая история, а линия прогноза рисуется только на выбранных проверочных неделях.

Store ID в исходном Kaggle-датасете — единственные доступные идентификаторы магазинов. Поэтому используются реальные ID (`Store 1`, `Store 2`, …), без искусственных названий.


## 1. Среда выполнения, SparkSession и пути

In [ ]:
import os
import re
import json
import socket
import shutil
import warnings
from pathlib import Path
from datetime import datetime, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
)
from pyspark.ml.evaluation import RegressionEvaluator

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

SEED = 42
TASK = 'walmart'
TITLE = 'Walmart Weekly Sales · Spark ML · chronological holdout'
TARGET = 'Weekly_Sales'
HERO_METRIC = 'mape'
TARGET_UNIT = '$'
HOLDOUT_FRACTION = 0.20
MAX_JSON_PREDICTIONS = 10000  # полный Walmart holdout обычно около 1.3k строк


def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for parent in (path, *path.parents):
        if (parent / 'requirements.txt').exists():
            return parent
    return Path.cwd().resolve()


def select_spark_master():
    requested = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077').strip()
    if requested.startswith('spark://'):
        endpoint = requested[len('spark://'):]
        host, _, port_text = endpoint.rpartition(':')
        try:
            with socket.create_connection((host, int(port_text)), timeout=1.5):
                print(f'Spark master доступен: {requested}')
                return requested
        except Exception as exc:
            print(f'Spark master недоступен ({type(exc).__name__}: {exc}); fallback на local[2].')
            return 'local[2]'
    return requested or 'local[2]'


PROJECT = find_project_root()
MODEL_DIR = PROJECT / 'models' / 'regression' / TASK
ART_DIR = PROJECT / 'artifacts' / 'regression' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

SPARK_MASTER = select_spark_master()
remote = SPARK_MASTER.startswith('spark://')
builder = (
    SparkSession.builder
    .appName('ml-regression-walmart-chronological')
    .master(SPARK_MASTER)
    .config('spark.driver.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.executor.memory', '6g' if remote else '2g')
    .config('spark.executor.memoryOverhead', '768m' if remote else '512m')
    .config('spark.executor.cores', '4' if remote else '2')
    .config('spark.cores.max', '4' if remote else '2')
    .config('spark.default.parallelism', '8' if remote else '4')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.ansi.enabled', 'false')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.fallback.enabled', 'true')
)
if Path('/opt/spark/events').is_dir():
    builder = builder.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', 'file:///opt/spark/events')
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print('Project      :', PROJECT)
print('Spark        :', spark.version)
print('Master       :', spark.sparkContext.master)
print('Artifacts dir:', ART_DIR)


## 2. Загрузка и очистка исходного Walmart CSV

In [ ]:
import kagglehub

local_candidates = [
    PROJECT / 'data' / 'walmart' / 'Walmart.csv',
    PROJECT / 'data' / 'walmart' / 'walmart.csv',
    PROJECT / 'data' / 'Walmart.csv',
    PROJECT / 'data' / 'walmart_sales.csv',
]
csv_path = next((p for p in local_candidates if p.exists()), None)
DATASET_NAME = 'Kaggle Walmart Store Sales'

if csv_path is None:
    try:
        downloaded = Path(kagglehub.dataset_download('talhaanjum0/walmart-dataset'))
        candidates = list(downloaded.rglob('*.csv'))
        preferred = [p for p in candidates if 'walmart' in p.name.lower()]
        candidates = preferred or sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
        csv_path = candidates[0] if candidates else None
    except Exception as exc:
        print('Kaggle download не удался:', repr(exc))

if csv_path is None or not Path(csv_path).exists():
    raise FileNotFoundError(
        'Не найден Walmart CSV. Поместите файл в data/walmart/Walmart.csv '
        'или настройте доступ kagglehub к talhaanjum0/walmart-dataset.'
    )

print('CSV source:', csv_path)
raw_pdf = pd.read_csv(csv_path)
raw_pdf.columns = [str(c).replace('\ufeff', '').strip() for c in raw_pdf.columns]


def norm_col(name):
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

aliases = {
    'store': 'Store', 'storeid': 'Store',
    'date': 'Date',
    'weeklysales': 'Weekly_Sales', 'sales': 'Weekly_Sales',
    'holidayflag': 'Holiday_Flag', 'isholiday': 'Holiday_Flag',
    'temperature': 'Temperature', 'fuelprice': 'Fuel_Price',
    'cpi': 'CPI', 'unemployment': 'Unemployment',
}
rename_map = {}
for col in raw_pdf.columns:
    canonical = aliases.get(norm_col(col))
    if canonical and canonical not in raw_pdf.columns:
        rename_map[col] = canonical
raw_pdf = raw_pdf.rename(columns=rename_map)

required = ['Store', 'Date', TARGET]
missing = [col for col in required if col not in raw_pdf.columns]
if missing:
    raise ValueError(f'Не найдены обязательные колонки {missing}. В файле есть: {list(raw_pdf.columns)}')

# Учитываем оба типовых формата исходного набора: d-m-Y и m/d/Y.
date_text = raw_pdf['Date'].astype(str).str.strip()
parsed_date = pd.to_datetime(date_text, format='%d-%m-%Y', errors='coerce')
parsed_date = parsed_date.fillna(pd.to_datetime(date_text, format='%Y-%m-%d', errors='coerce'))
parsed_date = parsed_date.fillna(pd.to_datetime(date_text, format='%m/%d/%Y', errors='coerce'))
parsed_date = parsed_date.fillna(pd.to_datetime(date_text, format='mixed', errors='coerce'))
raw_pdf['Date'] = parsed_date

for col in ['Store', TARGET, 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    if col not in raw_pdf.columns and col == 'Holiday_Flag':
        raw_pdf[col] = 0
    if col in raw_pdf.columns:
        raw_pdf[col] = pd.to_numeric(raw_pdf[col], errors='coerce')

raw_pdf = raw_pdf.dropna(subset=['Store', 'Date', TARGET]).copy()
raw_pdf = raw_pdf[raw_pdf[TARGET].map(np.isfinite)].copy()
raw_pdf['Store'] = raw_pdf['Store'].astype(int)
raw_pdf[TARGET] = raw_pdf[TARGET].astype(float)
raw_pdf['Holiday_Flag'] = raw_pdf['Holiday_Flag'].fillna(0).astype(float)

# Временные признаки создаются только из календаря: они доступны при backtest на будущих датах.
raw_pdf['year'] = raw_pdf['Date'].dt.year.astype(float)
raw_pdf['month'] = raw_pdf['Date'].dt.month.astype(float)
raw_pdf['week'] = raw_pdf['Date'].dt.isocalendar().week.astype(float)

candidate_features = [
    'Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'year', 'month', 'week',
]
FEATURES = [col for col in candidate_features if col in raw_pdf.columns]
if not FEATURES:
    raise ValueError('После очистки не осталось признаков для обучения.')

# Задаём границу времени до импутации и используем только ранний train-отрезок
# для оценки медиан. Так при заполнении пропусков статистика holdout не протекает в train.
known_dates_for_imputation = sorted(raw_pdf['Date'].dropna().unique().tolist())
if len(known_dates_for_imputation) < 14:
    raise ValueError(f'Для горизонта 13 недель нужно минимум 14 недель, найдено {len(known_dates_for_imputation)}.')
imputation_split_index = int(np.floor(len(known_dates_for_imputation) * (1.0 - HOLDOUT_FRACTION)))
imputation_split_index = max(1, min(imputation_split_index, len(known_dates_for_imputation) - 13))
imputation_test_start = known_dates_for_imputation[imputation_split_index]
train_mask_for_imputation = raw_pdf['Date'] < imputation_test_start

for col in FEATURES:
    raw_pdf[col] = pd.to_numeric(raw_pdf[col], errors='coerce')
    train_median = raw_pdf.loc[train_mask_for_imputation, col].median()
    if not np.isfinite(train_median):
        train_median = raw_pdf[col].median()
    if not np.isfinite(train_median):
        train_median = 0.0
    raw_pdf[col] = raw_pdf[col].fillna(float(train_median))

raw_pdf = raw_pdf.sort_values(['Date', 'Store']).drop_duplicates(['Date', 'Store'], keep='last').reset_index(drop=True)
raw_pdf['_row_id'] = np.arange(len(raw_pdf), dtype=np.int64)
raw_pdf['Date'] = raw_pdf['Date'].dt.date

clean_pdf = raw_pdf[['_row_id', 'Date', 'Store', *FEATURES, TARGET]].copy()
clean_pdf['Store'] = clean_pdf['Store'].astype(float)
for col in FEATURES + [TARGET]:
    clean_pdf[col] = clean_pdf[col].astype(float)

# DataFrame данных небольшой (Walmart classic ~6.4k строк), но fit/transform/metrics — в Spark ML.
data_sdf = spark.createDataFrame(clean_pdf.to_dict(orient='records')).cache()
N_SAMPLES = data_sdf.count()
N_STORES = data_sdf.select('Store').distinct().count()
N_DATES = data_sdf.select('Date').distinct().count()
print(f'Dataset: {DATASET_NAME} | rows={N_SAMPLES:,} | stores={N_STORES} | weekly dates={N_DATES}')
print('Features:', FEATURES)
print('Store IDs:', [int(r['Store']) for r in data_sdf.select('Store').distinct().orderBy('Store').collect()])
data_sdf.select(TARGET).describe().show()


## 3. EDA и хронологическое разбиение train / test

In [ ]:
# EDA отображается по фактическому датасету, без искусственных рядов.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(clean_pdf[TARGET], bins=45, edgecolor='white')
axes[0].set_title('Weekly Sales · distribution')
axes[0].set_xlabel('Weekly_Sales ($)')
axes[1].plot(
    clean_pdf.groupby('Date', as_index=False)[TARGET].sum()['Date'],
    clean_pdf.groupby('Date', as_index=False)[TARGET].sum()[TARGET],
    linewidth=1.2,
)
axes[1].set_title('Total weekly sales across stores')
axes[1].set_xlabel('Week')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight')
plt.show()

all_dates = sorted(pd.to_datetime(pd.Series(clean_pdf['Date'])).dt.date.unique().tolist())
if len(all_dates) < 14:
    raise ValueError(f'Для выбранных горизонтов нужно минимум 14 недель, в данных только {len(all_dates)}.')
cut_index = int(np.floor(len(all_dates) * (1.0 - HOLDOUT_FRACTION)))
cut_index = max(1, min(cut_index, len(all_dates) - 13))
TEST_START_DATE = all_dates[cut_index]

train_df = data_sdf.filter(F.col('Date') < F.lit(TEST_START_DATE)).cache()
test_df = data_sdf.filter(F.col('Date') >= F.lit(TEST_START_DATE)).cache()
N_TRAIN = train_df.count()
N_TEST = test_df.count()
N_TEST_DATES = test_df.select('Date').distinct().count()

if N_TRAIN == 0 or N_TEST == 0:
    raise ValueError('Получилось пустое train/test разбиение. Проверьте количество дат в датасете.')

print('Train: Date <', TEST_START_DATE, '| rows:', N_TRAIN)
print('Test : Date >=', TEST_START_DATE, '| rows:', N_TEST, '| dates:', N_TEST_DATES)
print('Важно: тестовые даты строго позже всех train-дат.')
assert train_df.agg(F.max('Date').alias('max_train')).first()['max_train'] < TEST_START_DATE


## 4. Метрики и набор Spark ML моделей

In [ ]:
def make_estimator(name):
    if name == 'Linear':
        return LinearRegression(featuresCol='features', labelCol=TARGET, predictionCol='prediction', maxIter=50, regParam=0.0)
    if name == 'Ridge':
        return LinearRegression(featuresCol='features', labelCol=TARGET, predictionCol='prediction', maxIter=50, regParam=0.1)
    if name == 'DecisionTree':
        return DecisionTreeRegressor(featuresCol='features', labelCol=TARGET, predictionCol='prediction', maxDepth=8, maxBins=32, minInstancesPerNode=5, seed=SEED)
    if name == 'RandomForest':
        return RandomForestRegressor(featuresCol='features', labelCol=TARGET, predictionCol='prediction', numTrees=60, maxDepth=8, maxBins=32, minInstancesPerNode=5, subsamplingRate=0.85, seed=SEED)
    if name == 'GradBoost':
        return GBTRegressor(featuresCol='features', labelCol=TARGET, predictionCol='prediction', maxIter=45, maxDepth=4, maxBins=32, stepSize=0.05, subsamplingRate=0.85, seed=SEED)
    raise KeyError(f'Unknown model: {name}')


def make_pipeline(name):
    stages = []
    # Для линейных моделей Store кодируется как категория, чтобы не предполагать
    # линейную зависимость продаж от числового ID магазина.
    use_store_ohe = 'Store' in FEATURES and name in {'Linear', 'Ridge'}
    if use_store_ohe:
        stages.append(StringIndexer(inputCol='Store', outputCol='Store_index', handleInvalid='keep'))
        stages.append(OneHotEncoder(inputCols=['Store_index'], outputCols=['Store_ohe'], handleInvalid='keep', dropLast=True))
        assembler_inputs = [col for col in FEATURES if col != 'Store'] + ['Store_ohe']
    else:
        assembler_inputs = list(FEATURES)
    stages.append(VectorAssembler(inputCols=assembler_inputs, outputCol='features', handleInvalid='error'))
    stages.append(make_estimator(name))
    return Pipeline(stages=stages)


def to_float(value, default=0.0):
    try:
        result = float(value)
        return result if np.isfinite(result) else float(default)
    except (TypeError, ValueError):
        return float(default)


def metrics_for(prediction_df):
    # MAPE исключает строки с нулевым фактом; для классического Walmart target положительный.
    enriched = (prediction_df
        .withColumn('_error', F.col('prediction') - F.col(TARGET))
        .withColumn('_abs_error', F.abs(F.col('_error')))
        .withColumn('_ape', F.when(F.abs(F.col(TARGET)) > F.lit(1e-9), F.abs(F.col('_error')) / F.abs(F.col(TARGET)) * 100.0)))
    row = enriched.agg(
        F.count('*').alias('n'),
        F.avg('_abs_error').alias('mae'),
        F.sqrt(F.avg(F.pow(F.col('_error'), 2))).alias('rmse'),
        F.avg('_ape').alias('mape'),
        F.percentile_approx('_abs_error', 0.5, 10000).alias('medae'),
        F.max('_abs_error').alias('max_error'),
        F.avg('_error').alias('bias'),
        F.var_pop('_error').alias('error_var'),
        F.var_pop(TARGET).alias('target_var'),
        F.avg(F.when(F.col(TARGET) > 0, F.pow(F.log1p(F.greatest(F.col('prediction'), F.lit(0.0))) - F.log1p(F.col(TARGET)), 2))).alias('log_sq_error'),
    ).first().asDict()
    r2_eval = RegressionEvaluator(labelCol=TARGET, predictionCol='prediction', metricName='r2').evaluate(prediction_df)
    target_var = to_float(row.get('target_var'))
    error_var = to_float(row.get('error_var'))
    metrics = {
        'n': int(row.get('n') or 0),
        'mae': to_float(row.get('mae')),
        'rmse': to_float(row.get('rmse')),
        'mape': to_float(row.get('mape')),
        'medae': to_float(row.get('medae')),
        'max_error': to_float(row.get('max_error')),
        'bias': to_float(row.get('bias')),
        'r2': to_float(r2_eval),
        'evs': (1.0 - error_var / target_var) if target_var > 1e-12 else 0.0,
        'rmsle': float(np.sqrt(max(0.0, to_float(row.get('log_sq_error'))))),
    }
    return metrics


# Baseline: медиана продаж по каждому магазину, оцененная исключительно на train.
TRAIN_MEDIAN = to_float(train_df.agg(F.percentile_approx(TARGET, 0.5, 10000).alias('m')).first()['m'])
store_medians = train_df.groupBy('Store').agg(F.percentile_approx(TARGET, 0.5, 10000).alias('_store_median'))
baseline_predictions = (test_df.join(store_medians, on='Store', how='left')
    .withColumn('prediction', F.coalesce(F.col('_store_median'), F.lit(TRAIN_MEDIAN)))
    .drop('_store_median'))

MODEL_NAMES = ['BaselineMedian', 'Linear', 'Ridge', 'DecisionTree', 'RandomForest', 'GradBoost']
fitted = {}
prediction_frames = {}
results = {}

baseline_frame = baseline_predictions.select('_row_id', 'Date', 'Store', TARGET, 'prediction').cache()
prediction_frames['BaselineMedian'] = baseline_frame
results['BaselineMedian'] = metrics_for(baseline_frame)
print('BaselineMedian:', {k: round(v, 4) for k, v in results['BaselineMedian'].items() if k != 'n'})

for model_name in MODEL_NAMES[1:]:
    print(f'\nTraining {model_name} ...')
    pipeline = make_pipeline(model_name)
    model = pipeline.fit(train_df)
    fitted[model_name] = model
    pred_df = (model.transform(test_df)
        .select('_row_id', 'Date', 'Store', TARGET, 'prediction')
        .cache())
    prediction_frames[model_name] = pred_df
    results[model_name] = metrics_for(pred_df)
    print(model_name, {k: round(v, 4) for k, v in results[model_name].items() if k != 'n'})

# Ранжирование только по holdout; одинаковый временной отрезок для всех моделей.
best_name = min(results, key=lambda name: results[name][HERO_METRIC])
best_by_rmse = min(results, key=lambda name: results[name]['rmse'])
print('\nBest by MAPE:', best_name, round(results[best_name]['mape'], 4), '%')
print('Best by RMSE:', best_by_rmse, round(results[best_by_rmse]['rmse'], 2), '$')


## 5. Диагностики и сравнение моделей

In [ ]:
# Один общий порядок строк гарантирует точное выравнивание факта и predictions[] в JSON.
export_pdf = (test_df.orderBy('Date', 'Store')
    .select('_row_id', 'Date', 'Store', *FEATURES, TARGET)
    .toPandas())
export_pdf['Date'] = pd.to_datetime(export_pdf['Date'])
if len(export_pdf) > MAX_JSON_PREDICTIONS:
    raise ValueError(f'Holdout содержит {len(export_pdf)} строк; лимит JSON {MAX_JSON_PREDICTIONS} меньше выборки.')

export_predictions = {}
for model_name in MODEL_NAMES:
    pred_pdf = (prediction_frames[model_name]
        .select('_row_id', 'prediction')
        .toPandas()
        .set_index('_row_id')
        .reindex(export_pdf['_row_id']))
    if pred_pdf['prediction'].isna().any():
        raise ValueError(f'Прогноз {model_name} нельзя выровнять по _row_id: есть пропуски.')
    export_predictions[model_name] = pred_pdf['prediction'].astype(float).to_numpy()

best_y = export_pdf[TARGET].astype(float).to_numpy()
best_pred = export_predictions[best_name]
residuals = best_pred - best_y

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(best_y, best_pred, s=12, alpha=0.45)
lo = float(min(np.min(best_y), np.min(best_pred)))
hi = float(max(np.max(best_y), np.max(best_pred)))
axes[0].plot([lo, hi], [lo, hi], '--', linewidth=1, color='black')
axes[0].set_xlabel('Actual Weekly Sales ($)')
axes[0].set_ylabel('Predicted Weekly Sales ($)')
axes[0].set_title(f'Actual vs Predicted · {best_name}')
axes[1].scatter(best_pred, residuals, s=12, alpha=0.45)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted ($)')
axes[1].set_ylabel('Prediction − actual ($)')
axes[1].set_title('Residuals · chronological holdout')
plt.tight_layout()
plt.savefig(ART_DIR / 'actual_vs_predicted.png', bbox_inches='tight')
plt.savefig(ART_DIR / 'residuals.png', bbox_inches='tight')
plt.show()

metrics_pdf = pd.DataFrame(results).T.reset_index(names='model').sort_values('mape')
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(metrics_pdf['model'], metrics_pdf['mape'])
ax.set_ylabel('MAPE (%)')
ax.set_title('Model comparison on chronological holdout · lower is better')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(ART_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()

# Подсчёт per-store ошибок на holdout для победителя; фронтенд пересчитывает MAPE для любой выбранной модели.
best_store = (prediction_frames[best_name]
    .withColumn('_ae', F.abs(F.col('prediction') - F.col(TARGET)))
    .withColumn('_ape', F.when(F.abs(F.col(TARGET)) > F.lit(1e-9), F.col('_ae') / F.abs(F.col(TARGET)) * 100.0))
    .groupBy('Store')
    .agg(F.count('*').alias('n_weeks'), F.avg(TARGET).alias('avg_test_sales'), F.avg('_ae').alias('mae'),
         F.sqrt(F.avg(F.pow(F.col('prediction') - F.col(TARGET), 2))).alias('rmse'), F.avg('_ape').alias('mape'))
    .orderBy('Store').toPandas())
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(best_store['Store'].astype(int).astype(str), best_store['mape'])
ax.set_xlabel('Real Store ID')
ax.set_ylabel('MAPE (%)')
ax.set_title(f'Per-store MAPE · {best_name} · time holdout')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(ART_DIR / 'mape_per_store.png', bbox_inches='tight')
plt.show()


## 6. Экспорт `report.json`, рядов по каждому магазину и метрик

In [ ]:
def iso_date(value):
    if pd.isna(value):
        return None
    return pd.Timestamp(value).strftime('%Y-%m-%d')


def clean_float_list(values):
    return [to_float(value) for value in values]


def spark_stats_for(frame, columns):
    expressions = []
    for col in columns:
        expressions.extend([
            F.min(F.col(col)).alias(f'{col}__min'),
            F.max(F.col(col)).alias(f'{col}__max'),
            F.avg(F.col(col)).alias(f'{col}__mean'),
            F.stddev_pop(F.col(col)).alias(f'{col}__std'),
            F.percentile_approx(F.col(col), 0.5, 10000).alias(f'{col}__median'),
        ])
    row = frame.agg(*expressions).first().asDict()
    return {
        col: {
            key: to_float(row.get(f'{col}__{key}'))
            for key in ['min', 'max', 'mean', 'std', 'median']
        }
        for col in columns
    }


feature_stats = spark_stats_for(data_sdf, FEATURES)
target_stats = spark_stats_for(data_sdf, [TARGET])[TARGET]

models_block = []
for model_name in MODEL_NAMES:
    metrics = {key: to_float(value) for key, value in results[model_name].items()}
    models_block.append({
        'name': model_name,
        'is_best': model_name == best_name,
        'is_best_by_rmse': model_name == best_by_rmse,
        'metrics': metrics,
        # predictions[i] соответствует data.y_test[i], data.dates_test[i] и data.stores_test[i]
        'predictions': clean_float_list(export_predictions[model_name]),
        'diagnostics': {
            'rmse_over_mae': to_float(metrics['rmse'] / max(metrics['mae'], 1e-9)),
            'mae_over_medae': to_float(metrics['mae'] / max(metrics['medae'], 1e-9)),
            'abs_bias': abs(to_float(metrics['bias'])),
        },
    })

# Все фактические недельные значения всех магазинов, без synthetic/demo ряда.
full_pdf = (data_sdf.select('Date', 'Store', TARGET).orderBy('Store', 'Date').toPandas())
series_by_store = {}
for store_id, group in full_pdf.groupby('Store', sort=True):
    series_by_store[str(int(store_id))] = {
        'dates': [iso_date(value) for value in group['Date'].tolist()],
        'sales': clean_float_list(group[TARGET].tolist()),
    }

# Метрики по всем магазинам из временного holdout для лучшей по MAPE модели.
per_store_rows = []
best_store_metrics = {
    int(row['Store']): row
    for _, row in best_store.iterrows()
}
full_store_stats = (data_sdf.groupBy('Store')
    .agg(F.avg(TARGET).alias('avg_sales'), F.percentile_approx(TARGET, 0.5, 10000).alias('median_sales'))
    .collect())
full_store_stats = {int(row['Store']): row.asDict() for row in full_store_stats}
for store_id in sorted(series_by_store, key=lambda key: int(key)):
    numeric_id = int(store_id)
    metric_row = best_store_metrics.get(numeric_id, {})
    all_row = full_store_stats.get(numeric_id, {})
    per_store_rows.append({
        'store': numeric_id,
        'n_weeks': int(metric_row.get('n_weeks', 0) or 0),
        # avg_sales — фактический средний уровень за полную доступную историю;
        # mae/rmse/mape — только ошибки на holdout.
        'avg_sales': to_float(all_row.get('avg_sales')),
        'median_sales': to_float(all_row.get('median_sales')),
        'mae': to_float(metric_row.get('mae')),
        'rmse': to_float(metric_row.get('rmse')),
        'mape': to_float(metric_row.get('mape')),
    })

# Только даты, у которых в настоящем CSV Holiday_Flag=1. Никакие будущие / искусственные праздники не добавляются.
date_holiday = {}
if 'Holiday_Flag' in clean_pdf.columns:
    holiday_pdf = clean_pdf.loc[clean_pdf['Holiday_Flag'] >= 0.5, ['Date']].drop_duplicates().sort_values('Date')
    for holiday_date in holiday_pdf['Date'].tolist():
        d = pd.Timestamp(holiday_date)
        mmdd = d.strftime('%m-%d')
        # Имена соответствуют праздничным периодам, представленным classic Walmart dataset.
        # Если CSV помечает дополнительную неделю, оставляем нейтральную подпись фактического holiday flag.
        if d.month == 2:
            label, icon = 'Super Bowl', '🏈'
        elif d.month == 9:
            label, icon = 'Labor Day', '🇺🇸'
        elif d.month == 11:
            label, icon = 'Thanksgiving', '🦃'
        elif d.month == 12:
            label, icon = 'Christmas', '🎄'
        else:
            label, icon = 'Holiday week', '★'
        actual_iso_date = d.strftime('%Y-%m-%d')
        date_holiday[actual_iso_date] = {'date': actual_iso_date, 'icon': icon, 'label': label}
holidays = [date_holiday[key] for key in sorted(date_holiday)]

# Справочник ошибок сохраняет исходную форму API и позволяет фронту пересчитать значения для выбранной модели.
export_actual = export_pdf[TARGET].astype(float).to_numpy()
error_by_best = best_pred - export_actual
closest = np.argsort(np.abs(error_by_best))[:min(12, len(error_by_best))]
sample_predictions = []
for idx in closest:
    feat_values = {feature: to_float(export_pdf.iloc[idx][feature]) for feature in FEATURES}
    error = to_float(error_by_best[idx])
    sample_predictions.append({
        'features': feat_values,
        'date': iso_date(export_pdf.iloc[idx]['Date']),
        'store': int(export_pdf.iloc[idx]['Store']),
        'actual': to_float(export_actual[idx]),
        'predicted': to_float(best_pred[idx]),
        'error': error,
        'abs_error': abs(error),
        'within_mae': bool(abs(error) < results[best_name]['mae']),
        'is_outlier': bool(abs(error) > 3 * results[best_name]['mae']),
    })

median_spread = [to_float(row['median_sales']) for row in full_store_stats.values()]
min_median = min(median_spread) if median_spread else 0.0
max_median = max(median_spread) if median_spread else 0.0
full_start = pd.Timestamp(full_pdf['Date'].min()).strftime('%Y-%m-%d')
full_end = pd.Timestamp(full_pdf['Date'].max()).strftime('%Y-%m-%d')

y_test = clean_float_list(export_actual)
dates_test = [iso_date(value) for value in export_pdf['Date'].tolist()]
stores_test = [int(value) for value in export_pdf['Store'].tolist()]

# Model importance: RandomForest uses the raw numeric FEATURES vector (unlike OHE linear models).
rf_stage = fitted['RandomForest'].stages[-1]
rf_importances = np.asarray(rf_stage.featureImportances.toArray(), dtype=float)
feature_importance = {
    'RandomForest': [
        {'name': name, 'value': to_float(value)}
        for name, value in sorted(zip(FEATURES, rf_importances), key=lambda pair: pair[1], reverse=True)
    ]
}

metric_names = ['mae', 'rmse', 'mape', 'r2', 'medae', 'max_error', 'evs', 'rmsle']
by_metric = {}
for metric in metric_names:
    valid = [(name, values.get(metric)) for name, values in results.items() if values.get(metric) is not None and np.isfinite(values.get(metric))]
    if valid:
        by_metric[metric] = (max(valid, key=lambda pair: pair[1])[0] if metric == 'r2' or metric == 'evs'
                             else min(valid, key=lambda pair: pair[1])[0])
by_metric['bias'] = min(results, key=lambda name: abs(results[name]['bias']))

report = {
    'meta': {
        'task': TASK,
        'title': TITLE,
        'dataset': DATASET_NAME,
        'target': TARGET,
        'target_unit': TARGET_UNIT,
        'target_format': 'money',
        'features': FEATURES,
        'seed': SEED,
        'generated_at': datetime.now().isoformat(),
        'n_samples': int(N_SAMPLES),
        'n_features': int(len(FEATURES)),
        'n_train': int(N_TRAIN),
        'n_test': int(N_TEST),
        'n_points_in_json': int(len(export_pdf)),
        'n_test_dates': int(N_TEST_DATES),
        'test_start_date': pd.Timestamp(TEST_START_DATE).strftime('%Y-%m-%d'),
        'test_end_date': full_end,
        'target_stats': target_stats,
        'split_strategy': 'chronological_holdout',
        'forecast_semantics': 'backtest using calendar/store and exogenous features observed for holdout dates',
        'narrative': {
            'context': 'Оцениваем недельные продажи Walmart на неделях, которые идут позже всех недель обучения.',
            'hero_metric': 'MAPE',
            'why_not_mae': 'MAPE сопоставляет относительную ошибку магазинов разного масштаба; MAE и RMSE отражают абсолютный промах в долларах.',
            'main_conclusion': 'Прогноз линии показывается только на выбранном горизонте последнего временного holdout; факт берётся из исходного CSV.',
        },
        'holidays': holidays,
        'store_spread': {
            'ratio_max_min': (max_median / min_median) if abs(min_median) > 1e-9 else None,
            'median_min': min_median,
            'median_max': max_median,
        },
        'series_span': {
            'start': full_start,
            'end': full_end,
            'n_weeks': int(data_sdf.select('Date').distinct().count()),
            'n_stores': int(N_STORES),
        },
    },
    'headline': {
        'best_model': best_name,
        'best_metric': HERO_METRIC,
        'best_metric_value': to_float(results[best_name][HERO_METRIC]),
        'verdict': f'Лучшая по MAPE на хронологическом holdout — {best_name}.',
        'runner_up_by_rmse': {'model': best_by_rmse, 'rmse': to_float(results[best_by_rmse]['rmse'])},
        'diagnostic_verdict': {
            'rmse_over_mae': to_float(results[best_name]['rmse'] / max(results[best_name]['mae'], 1e-9)),
            'mae_over_medae': to_float(results[best_name]['mae'] / max(results[best_name]['medae'], 1e-9)),
            'flag': 'large_errors_present' if results[best_name]['rmse'] / max(results[best_name]['mae'], 1e-9) > 1.5 else 'stable',
        },
    },
    'data': {
        'y_test': y_test,
        'dates_test': dates_test,
        'stores_test': stores_test,
        'dates_train': [iso_date(value) for value in train_df.select('Date').distinct().orderBy('Date').toPandas()['Date'].tolist()],
        'stores_train': [int(value) for value in train_df.select('Store').distinct().orderBy('Store').toPandas()['Store'].tolist()],
        'feature_names': FEATURES,
        'feature_stats': feature_stats,
        'n_test_full': int(N_TEST),
    },
    'calculator': {
        'target': TARGET,
        'target_unit': TARGET_UNIT,
        'features': [
            {'name': feature, 'min': feature_stats[feature]['min'], 'max': feature_stats[feature]['max'],
             'median': feature_stats[feature]['median'], 'default': feature_stats[feature]['median']}
            for feature in FEATURES
        ],
    },
    'models': models_block,
    'by_metric': by_metric,
    'residual_diagnostics': {
        'best_model': best_name,
        'n': len(residuals),
        'mean': to_float(np.mean(residuals)),
        'std': to_float(np.std(residuals)),
        'p95_abs': to_float(np.quantile(np.abs(residuals), 0.95)),
        'p99_abs': to_float(np.quantile(np.abs(residuals), 0.99)),
    },
    'feature_importance': feature_importance,
    'sample_predictions': sample_predictions,
    'charts': {
        'eda': 'eda.png',
        'actual_vs_predicted': 'actual_vs_predicted.png',
        'residuals': 'residuals.png',
        'model_comparison': 'model_comparison.png',
        'mape_per_store': 'mape_per_store.png',
    },
    'artifacts': {
        'metrics_csv': 'metrics_all_models.csv',
        'predictions_csv': 'predictions_test.csv',
        'model': str((MODEL_DIR / f'{TASK}_spark_model').relative_to(PROJECT)).replace('\\', '/') if best_name != 'BaselineMedian' else None,
        'model_meta': str((MODEL_DIR / f'{TASK}_model_meta.json').relative_to(PROJECT)).replace('\\', '/'),
    },
    'series_by_store': series_by_store,
    'per_store': per_store_rows,
}

report_path = ART_DIR / 'report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

metrics_rows = []
for name, values in results.items():
    metrics_rows.append({'model': name, **{k: to_float(v) for k, v in values.items()}})
pd.DataFrame(metrics_rows).sort_values('mape').to_csv(ART_DIR / 'metrics_all_models.csv', index=False)

predictions_export = export_pdf[['_row_id', 'Date', 'Store', TARGET]].copy()
predictions_export['Date'] = predictions_export['Date'].dt.strftime('%Y-%m-%d')
for name in MODEL_NAMES:
    predictions_export[f'prediction_{name}'] = export_predictions[name]
predictions_export.to_csv(ART_DIR / 'predictions_test.csv', index=False)

model_path = MODEL_DIR / f'{TASK}_spark_model'
if best_name != 'BaselineMedian':
    fitted[best_name].write().overwrite().save(str(model_path))
    # Reload smoke-check ensures the saved Spark model can be loaded again.
    reloaded = PipelineModel.load(str(model_path))
    reloaded_count = reloaded.transform(test_df.limit(5)).count()
    if reloaded_count == 0:
        raise RuntimeError('Сохраненная Spark-модель не прошла smoke-check.')

model_meta = {
    'task': TASK,
    'dataset': DATASET_NAME,
    'format': 'pyspark.ml.PipelineModel' if best_name != 'BaselineMedian' else 'store_median_baseline',
    'model': best_name,
    'model_path': str(model_path.relative_to(PROJECT)).replace('\\', '/') if best_name != 'BaselineMedian' else None,
    'features': FEATURES,
    'target': TARGET,
    'target_unit': TARGET_UNIT,
    'n_samples': int(N_SAMPLES),
    'seed': SEED,
    'spark_version': spark.version,
    'split_strategy': 'chronological_holdout',
    'test_start_date': pd.Timestamp(TEST_START_DATE).strftime('%Y-%m-%d'),
    'metrics': results[best_name],
    'all_models_metrics': results,
    'saved_at': datetime.now().isoformat(),
}
(MODEL_DIR / f'{TASK}_model_meta.json').write_text(json.dumps(model_meta, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

# Обновляем только Walmart-запись в общем artifacts/index.json, не затрагивая остальные задачи.
index_path = PROJECT / 'artifacts' / 'index.json'
index_path.parent.mkdir(parents=True, exist_ok=True)
if index_path.exists():
    try:
        index = json.loads(index_path.read_text(encoding='utf-8'))
    except Exception:
        index = {'generated_at': None, 'tasks': []}
else:
    index = {'generated_at': None, 'tasks': []}
index.setdefault('tasks', [])
index['tasks'] = [item for item in index['tasks'] if item.get('id') != TASK]
index['tasks'].append({
    'id': TASK,
    'title': TITLE,
    'kind': 'regression',
    'best_model': best_name,
    'hero_metric': 'MAPE',
    'hero_value': to_float(results[best_name]['mape']),
    'unit': '%',
    'report': 'artifacts/regression/walmart/report.json',
    'charts_dir': 'artifacts/regression/walmart',
})
index['tasks'] = sorted(index['tasks'], key=lambda item: item.get('id', ''))
index['generated_at'] = datetime.now().isoformat()
index_path.write_text(json.dumps(index, indent=2, ensure_ascii=False), encoding='utf-8')

print('\nREPORT:', report_path)
print('INDEX :', index_path)
print('Best model:', best_name, '| MAPE:', round(results[best_name]['mape'], 4), '%')
print('Time holdout:', pd.Timestamp(TEST_START_DATE).strftime('%Y-%m-%d'), '→', full_end)
print('Stores in report:', len(series_by_store), '| holdout rows:', len(export_pdf))
print('Prediction alignment:', len(y_test), 'actuals and', len(export_predictions[best_name]), 'best-model predictions')
print('Artifacts:')
for artifact in sorted(ART_DIR.iterdir()):
    print(f'  {artifact.name:32s} {artifact.stat().st_size:>10,} bytes')


## 7. Контроль результата и ограничения интерпретации

In [ ]:
assert len(report['data']['y_test']) == len(report['data']['dates_test']) == len(report['data']['stores_test'])
assert all(len(model['predictions']) == len(report['data']['y_test']) for model in report['models'])
assert len(report['series_by_store']) == report['meta']['series_span']['n_stores']
assert report['meta']['split_strategy'] == 'chronological_holdout'
print('Task:', TASK)
print('Spark:', spark.version)
print('Train rows:', N_TRAIN, '| holdout rows:', N_TEST, '| holdout weeks:', N_TEST_DATES)
print('Stores:', N_STORES)
print('Best MAPE:', best_name, f"{results[best_name]['mape']:.3f}%")
print('All schema/alignment checks passed.')
print('\nВажно: это backtest на доступных исторических неделях, а не прогноз после последней даты датасета.')
print('Температура, CPI, Fuel_Price и Unemployment на holdout берутся из фактических строк CSV; при реальном запуске наперёд потребуются их прогнозы/плановые значения.')
